# Sprint 10. Mejora controlada de ResNet50 con AdamW

Clasificación de defectos de palta Hass: PICADURA y RUSSET. Se compara una única estrategia de regularización contra la referencia fija del Sprint 09 (`RN50_FECHA260423_s42_01`), sin reemplazar ni modificar el entrenamiento anterior.

**Hipótesis:** aplicar decaimiento desacoplado de pesos con AdamW podría reducir el sobreajuste y mejorar la capacidad de clasificación. Es un experimento, **no una mejora demostrada**.

Se mantienen exactamente los mismos conjuntos del Sprint 09: TRAIN revisado 529 imágenes, VALIDATION original 127 imágenes y grupo de fecha `260423` con 43 imágenes reservadas y excluidas del entrenamiento. Se conserva ResNet50 ImageNet, cabeza nueva, D1, augmentación, dropout 0.35, batch 1, 8 épocas máximas por etapa, learning rates, semilla 42 y selección del checkpoint por `val_loss`.

**Único cambio experimental:** Adam de ambas etapas pasa a AdamW con `weight_decay=0.01`, sin cambiar learning rates. Así se prueba una regularización concreta sin una nueva búsqueda de hiperparámetros.

La comparación y selección se basan exclusivamente en `validation_original`. **NO se calculan nuevas predicciones ni métricas de `evaluacion_exploratoria_260423` o TEST.** El grupo 260423 ya se evaluó en Sprint 09; consultarlo repetidamente para decidir modificaciones produciría sesgo de selección. Las métricas nuevas son evidencia de desarrollo, no de generalización independiente.


## Uso, control experimental y seguridad

1. Crear una rama dedicada `sprint-10-regularizacion` desde `sprint-09-validacion-por-fecha`. Conservar intacto `08_ResNet50_Validacion_Por_Fecha.ipynb`.
2. Copiar este archivo a `6. entrenamiento/09_Regularizacion_Controlada_ResNet50.ipynb` y abrir en VS Code con `palta_gpu (CUDA)`.
3. Ejecutar las primeras cuatro celdas de código: rutas, auditoría, preparación D1 y definición de funciones. No entrenan.
4. La quinta celda contiene `EJECUTAR_EXPERIMENTO = False`; cambiar a `True` una sola vez al iniciar el entrenamiento. Volver a `False` y guardar al terminar. No usar `Run All`.
5. La sexta celda compara métricas **solo de VALIDATION** contra el Sprint 09, sin volver a entrenar.

Checkpoint nuevo y resultados en `7. resultados/regularizacion_controlada/RN50_ADAMW_s42_01/`. Si existe esa carpeta, el entrenamiento se detiene sin sobrescribirla. Conservar checkpoints locales excluidos de Git.


## 1. Rutas y configuración

In [1]:
from pathlib import Path, PureWindowsPath
from collections import Counter
import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "README.md").is_file()
             and (p / "utilidades_imagenes.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook dentro de PROYECTO_TI_2 en VS Code/WSL.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from utilidades_imagenes import encontrar_raiz, resolver_dataset, cargar_rgb, ajustar_tamano

RAIZ = encontrar_raiz()
DATASET = resolver_dataset(RAIZ, "../DATASET_PALTA")
MANIFIESTO_RUTA = RAIZ / "7. resultados/validacion_por_fecha/manifiesto_260423.csv"
EDA = RAIZ / "7. resultados/eda"
AUDITORIA_ORIGEN = RAIZ / "7. resultados/entrenamiento/FT_D1_PHOTO_s42_01/auditoria.json"
SALIDA = RAIZ / "7. resultados/regularizacion_controlada"
SEMILLA = 42
TAMANO = (256, 512)  # alto, ancho
TAMANO_ENTRADA = TAMANO
CLASES = ["PICADURA", "RUSSET"]
NOMBRE_CORRIDA = f"RN50_ADAMW_s{SEMILLA}_01"

# Se define antes de importar TensorFlow.
os.environ.setdefault("KERAS_HOME", str(RAIZ / ".cache" / "keras"))

print("Proyecto:", RAIZ)
print("Dataset:", DATASET)
print("Manifiesto:", MANIFIESTO_RUTA.relative_to(RAIZ))
print("Corrida:", NOMBRE_CORRIDA)
print("Python:", sys.executable)

print("Cambio controlado: AdamW, weight_decay=0.01 en cabeza y fine-tuning.")


Proyecto: /mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2/PROYECTO_TI_2
Dataset: /mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2/DATASET_PALTA
Manifiesto: 7. resultados/validacion_por_fecha/manifiesto_260423.csv
Corrida: RN50_ADAMW_s42_01
Python: /home/tci_auto/.venvs/palta_gpu/bin/python
Cambio controlado: AdamW, weight_decay=0.01 en cabeza y fine-tuning.


## 2. Auditoría y separación fija

Verifica manifiesto EDA, las cuatro exclusiones originales, la identidad del dataset y ausencia de duplicados exactos entre las particiones. Conservar la separación del Sprint 09, sin mover archivos ni usar las 43 imágenes en el entrenamiento.


In [2]:
def hash_archivo(ruta):
    calculo = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculo.update(bloque)
    return calculo.hexdigest()


def firma_tabla(tabla):
    texto = "\n".join(tabla["archivo"] + ":" + tabla["sha256"])
    return hashlib.sha256(texto.encode()).hexdigest()


def leer_auditoria():
    carpeta = RAIZ / "7. resultados" / "eda"
    resumen = json.loads((carpeta / "eda_resumen.json").read_text(encoding="utf-8-sig"))
    inventario = pd.read_csv(carpeta / "eda_inventario.csv", encoding="utf-8-sig", keep_default_na=False)
    duplicados = pd.read_csv(carpeta / "eda_duplicados.csv", encoding="utf-8-sig", keep_default_na=False)
    columnas = {"archivo", "sha256", "sha_pixeles", "particion", "clase", "imagen", "error"}
    if not columnas.issubset(inventario.columns):
        raise ValueError("El inventario no contiene la auditoria necesaria.")
    if inventario.empty or inventario["archivo"].duplicated().any():
        raise ValueError("El inventario esta vacio o contiene rutas repetidas.")
    if len(inventario) != resumen["archivos"] or len(inventario) != resumen["imagenes"]:
        raise ValueError("Los conteos no coinciden. Revise el EDA.")
    if resumen["corruptas"] or inventario["error"].ne("").any():
        raise ValueError("El EDA registra errores de lectura.")
    if not inventario["imagen"].astype(str).str.lower().isin(["true", "1"]).all():
        raise ValueError("El inventario incluye archivos no reconocidos como imagenes.")
    if firma_tabla(inventario) != resumen["sha256_dataset"]:
        raise ValueError("La firma del inventario no coincide con el resumen.")
    for campo in ("sha256", "sha_pixeles"):
        if not inventario[campo].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
            raise ValueError(f"Hashes incompletos en {campo}.")
    for registro in inventario.itertuples(index=False):
        ruta = Path(registro.archivo)
        win = PureWindowsPath(registro.archivo)
        if (ruta.is_absolute() or win.drive or win.root or ".." in ruta.parts
                or "\\" in registro.archivo or len(ruta.parts) < 3):
            raise ValueError("El inventario debe usar rutas relativas dentro del dataset.")
        if ruta.parts[:2] != (registro.particion, registro.clase):
            raise ValueError(f"Ruta y etiqueta incoherentes: {registro.archivo}")
        if registro.particion not in {"train", "validation", "test"}:
            raise ValueError("Particion no reconocida.")
        if registro.clase not in {"PICADURA", "RUSSET"}:
            raise ValueError("Clase no reconocida.")

    requeridas = {"tipo", "archivo", "archivo_coincidente", "particion",
                  "particion_coincidente", "clase", "clase_coincidente"}
    if not requeridas.issubset(duplicados.columns):
        raise ValueError("El reporte de duplicados esta incompleto.")
    exactos = duplicados[duplicados["tipo"].isin(["bytes_exactos", "pixeles_exactos"])]
    indexado = inventario.set_index("archivo")
    pares_fuga, pares_conflicto, excluidos = set(), set(), set()
    for fila in exactos.itertuples(index=False):
        a, b = fila.archivo, fila.archivo_coincidente
        if a == b or a not in indexado.index or b not in indexado.index:
            raise ValueError("El reporte de duplicados contiene una ruta invalida.")
        ra, rb = indexado.loc[a], indexado.loc[b]
        if (ra.particion, rb.particion, ra.clase, rb.clase) != (
                fila.particion, fila.particion_coincidente, fila.clase, fila.clase_coincidente):
            raise ValueError("Duplicados e inventario no coinciden.")
        clave = "sha256" if fila.tipo == "bytes_exactos" else "sha_pixeles"
        if ra[clave] != rb[clave]:
            raise ValueError("El par reportado no coincide con los hashes del inventario.")
        par = tuple(sorted([a, b]))
        fuga, conflicto = ra.particion != rb.particion, ra.clase != rb.clase
        if fuga:
            pares_fuga.add(par)
        if conflicto:
            pares_conflicto.add(par)
        if fuga or conflicto:
            excluidos.update([a, b])
    if (len(pares_fuga) != resumen["pares_exactos_entre_particiones"]
            or len(pares_conflicto) != resumen["pares_exactos_etiquetas_distintas"]):
        raise ValueError("Los conteos de pares no coinciden con el resumen.")

    elegibles = inventario.loc[~inventario["archivo"].isin(excluidos)].copy()
    for clave in ("sha256", "sha_pixeles"):
        grupos = elegibles.groupby(clave)[["particion", "clase"]].nunique()
        if (grupos > 1).any().any():
            raise ValueError("Persisten duplicados entre particiones o clases. Revise el EDA.")
    return resumen, inventario, sorted(excluidos)


def verificar_auditoria(dataset, aprobar_exclusiones=False):
    resumen, inventario, excluidos = leer_auditoria()
    dataset = Path(dataset).resolve()
    rutas = set(inventario["archivo"])
    locales = {p.relative_to(dataset).as_posix() for p in dataset.rglob("*") if p.is_file()}
    if locales != rutas:
        raise RuntimeError("El contenido del dataset cambio. Revise las rutas y actualice el EDA.")
    total = len(inventario)
    for numero, registro in enumerate(inventario.itertuples(index=False), 1):
        ruta = (dataset / registro.archivo).resolve()
        if not ruta.is_relative_to(dataset):
            raise ValueError("Un enlace apunta fuera del dataset.")
        if hash_archivo(ruta) != registro.sha256:
            raise RuntimeError(f"Archivo distinto al auditado: {registro.archivo}")
        if numero % 100 == 0 or numero == total:
            print(f"Integridad: {numero}/{total}")
    if excluidos and not aprobar_exclusiones:
        raise RuntimeError(
            "Integridad correcta; deben aprobarse las exclusiones. "
            "No se entreno ni se modificaron imagenes."
        )
    elegibles = inventario.loc[~inventario["archivo"].isin(excluidos)].sort_values("archivo")
    return {
        "sha256_dataset": resumen["sha256_dataset"],
        "sha256_manifiesto": firma_tabla(elegibles),
        "exclusiones": excluidos,
        "archivos_originales": int(len(inventario)),
        "archivos_elegibles": int(len(elegibles)),
        "alcance": "Integridad exacta; independencia por fruto/lote pendiente de trazabilidad.",
    }

# La auditoría es la misma del entrenamiento original; la división por fecha es nueva.
for requisito in (MANIFIESTO_RUTA, AUDITORIA_ORIGEN,
                  EDA / "eda_inventario.csv", EDA / "eda_duplicados.csv",
                  EDA / "eda_resumen.json"):
    if not requisito.is_file():
        raise FileNotFoundError(f"Archivo necesario no encontrado: {requisito}")

referencia = json.loads(AUDITORIA_ORIGEN.read_text(encoding="utf-8"))
resumen_eda, inventario_eda, excluidos_eda = leer_auditoria()
if sorted(excluidos_eda) != sorted(referencia["exclusiones"]):
    raise ValueError("Las cuatro exclusiones no coinciden con el entrenamiento histórico.")
if (resumen_eda["sha256_dataset"] != referencia["sha256_dataset"] or
        firma_tabla(inventario_eda.loc[
            ~inventario_eda["archivo"].isin(excluidos_eda)
        ].sort_values("archivo")) != referencia["sha256_manifiesto"]):
    raise ValueError("La auditoría EDA no coincide con el protocolo del entrenamiento original.")

manifiesto = pd.read_csv(MANIFIESTO_RUTA, encoding="utf-8-sig", keep_default_na=False)
columnas = {"archivo", "clase", "conjunto"}
if not columnas.issubset(manifiesto.columns) or manifiesto.empty:
    raise ValueError("El manifiesto necesita archivo, clase y conjunto.")
if manifiesto["archivo"].duplicated().any():
    raise ValueError("El manifiesto contiene rutas repetidas.")

esperado = inventario_eda.loc[
    ~inventario_eda["archivo"].isin(excluidos_eda)
    & inventario_eda["particion"].isin(["train", "validation"]),
    ["archivo", "clase", "particion", "sha256", "sha_pixeles"],
].copy()
esperado["conjunto"] = np.where(
    esperado["particion"].eq("validation"), "validation_original",
    np.where(esperado["archivo"].str.contains(r"-260423-", regex=True),
             "evaluacion_exploratoria_260423", "train_revisado"),
)

comparar = ["archivo", "clase", "conjunto"]
a = manifiesto[comparar].sort_values(comparar).reset_index(drop=True)
b = esperado[comparar].sort_values(comparar).reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_dtype=False)

conteos = Counter(zip(manifiesto["conjunto"], manifiesto["clase"]))
conteos_esperados = {
    ("train_revisado", "PICADURA"): 474,
    ("train_revisado", "RUSSET"): 55,
    ("validation_original", "PICADURA"): 106,
    ("validation_original", "RUSSET"): 21,
    ("evaluacion_exploratoria_260423", "PICADURA"): 21,
    ("evaluacion_exploratoria_260423", "RUSSET"): 22,
}
if conteos != conteos_esperados or len(manifiesto) != 699:
    raise ValueError(f"Conteos no esperados: {dict(conteos)}")

# El EDA anterior permitía duplicados dentro de TRAIN. Al mover 43 TIFF a
# evaluación, debemos comprobar de nuevo los duplicados cruzados por bytes/píxeles.
con_hash = manifiesto.merge(
    esperado[["archivo", "sha256", "sha_pixeles"]],
    on="archivo", how="left", validate="one_to_one",
)
for clave in ("sha256", "sha_pixeles"):
    conflicto = con_hash.groupby(clave)["conjunto"].nunique()
    conflicto = conflicto[conflicto > 1]
    if not conflicto.empty:
        afectados = con_hash.loc[con_hash[clave].isin(conflicto.index),
                                ["archivo", "clase", "conjunto", clave]]
        print(afectados.to_string(index=False))
        raise RuntimeError(f"Hay duplicados exactos entre conjuntos según {clave}. No entrenar.")

for archivo in manifiesto["archivo"]:
    ruta = Path(archivo)
    win = PureWindowsPath(archivo)
    if ruta.is_absolute() or win.drive or win.root or ".." in ruta.parts or "\\" in archivo:
        raise ValueError(f"Ruta fuera de dataset: {archivo}")
    destino = (DATASET / archivo).resolve()
    if not destino.is_relative_to(DATASET.resolve()) or not destino.is_file():
        raise FileNotFoundError(f"Imagen no localizada dentro del dataset: {archivo}")

CONJUNTOS = {}
for nombre in ("train_revisado", "validation_original", "evaluacion_exploratoria_260423"):
    tabla = manifiesto.loc[manifiesto["conjunto"] == nombre, ["archivo", "clase"]].copy()
    tabla = tabla.sort_values("archivo").reset_index(drop=True)
    tabla["particion"] = tabla["archivo"].str.split("/").str[0]
    tabla["etiqueta"] = tabla["clase"].map({"PICADURA": 0, "RUSSET": 1}).astype("int32")
    CONJUNTOS[nombre] = tabla

print("Auditoría coherente. No hay duplicados exactos cruzados en la nueva división.")
display(pd.crosstab(manifiesto["conjunto"], manifiesto["clase"]))
print("Total TRAIN + VALIDATION + evaluación exploratoria:", len(manifiesto))
print("No se entrenó ningún modelo ni se calcularon predicciones de TEST.")


Auditoría coherente. No hay duplicados exactos cruzados en la nueva división.


clase,PICADURA,RUSSET
conjunto,,
evaluacion_exploratoria_260423,21,22
train_revisado,474,55
validation_original,106,21


Total TRAIN + VALIDATION + evaluación exploratoria: 699
No se entrenó ningún modelo ni se calcularon predicciones de TEST.


## 3. Balanceo D1 y preprocesamiento

La distribución sigue siendo 529 imágenes elegibles y 55 copias RUSSET (584 muestras efectivas por época). No se cambian las probabilidades de aumentos ni los `class_weight` de Sprint 09.


In [3]:
def plan_copias_russet(tabla_train, copias, semilla=SEMILLA):
    """Registra copias de los frutos RUSSET sin alterar el inventario."""
    if not isinstance(copias, int) or copias < 0 or copias > 3:
        raise ValueError("Use entre 0 y 3 copias por fruto.")
    russet = tabla_train.loc[tabla_train["clase"] == "RUSSET", "archivo"].tolist()
    filas = []
    for archivo in russet:
        for numero in range(1, copias + 1):
            clave = f"{semilla}|{archivo}|{numero}".encode("utf-8")
            entero = int.from_bytes(hashlib.sha256(clave).digest()[:8], "big")
            rng = np.random.default_rng(entero)
            filas.append({"archivo": archivo, "copia": numero, "etiqueta": 1,
                          "brillo": float(rng.uniform(0.92, 1.08)),
                          "contraste": float(rng.uniform(0.90, 1.10))})
    return pd.DataFrame(filas, columns=["archivo", "copia", "etiqueta", "brillo", "contraste"])


def aplicar_cambio_fotometrico(matriz, mascara, brillo, contraste):
    """Cambia la intensidad y el contraste sin mover pixeles ni modificar el relleno."""
    matriz = np.asarray(matriz, dtype=np.float32)
    mascara = np.asarray(mascara, dtype=bool)
    if matriz.ndim != 3 or matriz.shape[-1] != 3 or mascara.shape != matriz.shape[:2]:
        raise ValueError("Revise las dimensiones RGB y la mascara.")
    if not (0.85 <= brillo <= 1.15 and 0.85 <= contraste <= 1.15):
        raise ValueError("El cambio solicitado esta fuera del rango previsto.")
    resultado = matriz.copy()
    if mascara.any():
        pixeles = matriz[mascara]
        media = pixeles.mean(axis=0)
        resultado[mascara] = np.clip((pixeles - media) * contraste * brillo + media * brillo, 0, 255)
    return resultado




def pesos_para_etiquetas(etiquetas, clases):
    """Calcula class_weight con la distribucion efectiva de cada condicion."""
    from sklearn.utils.class_weight import compute_class_weight
    etiquetas = np.asarray(etiquetas, dtype=np.int32)
    valores = np.arange(len(clases), dtype=np.int32)
    if set(np.unique(etiquetas)) != set(valores):
        raise ValueError("El entrenamiento debe conservar ambas clases.")
    pesos = compute_class_weight("balanced", classes=valores, y=etiquetas)
    return {int(k): float(v) for k, v in zip(valores, pesos)}

BRILLO_EXTRA = (0.95, 1.05)
CONTRASTE_EXTRA = (0.95, 1.05)


def construir_manifiesto_entrenamiento(tabla_train, plan):
    base = tabla_train[["archivo", "etiqueta"]].copy()
    base["copia"] = 0
    base["brillo_d1"] = 1.0
    base["contraste_d1"] = 1.0
    extras = plan.rename(columns={"brillo": "brillo_d1", "contraste": "contraste_d1"})[
        ["archivo", "etiqueta", "copia", "brillo_d1", "contraste_d1"]
    ].copy()
    manifiesto = pd.concat([base, extras], ignore_index=True)
    if manifiesto["archivo"].isna().any():
        raise ValueError("El manifiesto incluye rutas vacias.")
    if not set(manifiesto["etiqueta"]) == {0, 1}:
        raise ValueError("El entrenamiento debe incluir PICADURA y RUSSET.")
    return manifiesto


def factores_para_visita(indice_global, archivo, semilla=SEMILLA):
    clave = f"{semilla}|{int(indice_global)}|{archivo}".encode("utf-8")
    entero = int.from_bytes(hashlib.sha256(clave).digest()[:8], "big")
    generador = np.random.default_rng(entero)
    brillo = float(generador.uniform(*BRILLO_EXTRA))
    contraste = float(generador.uniform(*CONTRASTE_EXTRA))
    return brillo, contraste


def crear_train_fotometrico(manifiesto, dataset, tamano=TAMANO, lote=1, semilla=SEMILLA, foto_extra=True):
    """Usa D1 en ambas etapas; ±5 % extra solo cuando foto_extra=True."""
    import tensorflow as tf
    if lote != 1:
        raise ValueError("Este protocolo utiliza lote de tamaño 1.")
    tamano = tuple(map(int, tamano))
    dataset = Path(dataset)
    if manifiesto.empty:
        raise ValueError("El entrenamiento esta vacio.")

    archivos = manifiesto["archivo"].astype(str).tolist()
    etiquetas = manifiesto["etiqueta"].to_numpy(dtype=np.int32)
    copias = manifiesto["copia"].to_numpy(dtype=np.int32)
    brillos = manifiesto["brillo_d1"].to_numpy(dtype=np.float32)
    contrastes = manifiesto["contraste_d1"].to_numpy(dtype=np.float32)

    secuencia = tf.data.Dataset.from_tensor_slices(
        (archivos, etiquetas, copias, brillos, contrastes)
    )
    secuencia = secuencia.shuffle(
        buffer_size=len(manifiesto), seed=semilla, reshuffle_each_iteration=True
    ).repeat().enumerate()

    def cargar_y_transformar(indice, ruta_relativa, etiqueta, copia, brillo_d1, contraste_d1):
        ruta_bytes = ruta_relativa.item() if hasattr(ruta_relativa, "item") else ruta_relativa
        ruta = ruta_bytes.decode("utf-8") if isinstance(ruta_bytes, bytes) else str(ruta_bytes)
        imagen = cargar_rgb(dataset / ruta)
        rgb, mascara = ajustar_tamano(imagen, tamano)
        rgb = np.asarray(rgb, dtype=np.float32)
        mascara = np.asarray(mascara, dtype=bool)
        if int(copia) > 0:
            rgb = aplicar_cambio_fotometrico(
                rgb, mascara, float(brillo_d1), float(contraste_d1)
            )
        if foto_extra:
            brillo, contraste = factores_para_visita(int(indice), ruta, semilla)
            rgb = aplicar_cambio_fotometrico(rgb, mascara, brillo, contraste)
        if not np.isfinite(rgb).all():
            raise ValueError(f"La imagen transformada contiene valores invalidos: {ruta}")
        return rgb.astype(np.float32), np.int32(etiqueta)

    def mapear(indice, registro):
        ruta, etiqueta, copia, brillo_d1, contraste_d1 = registro
        imagen, clase = tf.numpy_function(
            cargar_y_transformar,
            [indice, ruta, etiqueta, copia, brillo_d1, contraste_d1],
            Tout=[tf.float32, tf.int32]
        )
        imagen.set_shape((*tamano, 3))
        clase.set_shape(())
        return imagen, clase

    opciones = tf.data.Options()
    opciones.threading.private_threadpool_size = 2
    opciones.experimental_deterministic = True
    secuencia = secuencia.with_options(opciones)
    return secuencia.map(mapear, num_parallel_calls=1).batch(lote).prefetch(1)

# D1 se recalcula a partir de los 55 RUSSET disponibles en el nuevo TRAIN.
plan_d1 = plan_copias_russet(CONJUNTOS["train_revisado"], 1, SEMILLA)
train_efectivo = construir_manifiesto_entrenamiento(CONJUNTOS["train_revisado"], plan_d1)
pesos_efectivos = pesos_para_etiquetas(train_efectivo["etiqueta"].to_numpy(), CLASES)
if len(plan_d1) != 55 or len(train_efectivo) != 584:
    raise ValueError("El plan D1 debe generar 55 copias y 584 muestras por época.")
print("TRAIN original:", len(CONJUNTOS["train_revisado"]))
print("RUSSET sintéticos D1:", len(plan_d1))
print("TRAIN efectivo:", len(train_efectivo))
print("Class weight:", pesos_efectivos)


TRAIN original: 529
RUSSET sintéticos D1: 55
TRAIN efectivo: 584
Class weight: {0: 0.6160337552742616, 1: 2.6545454545454548}


## 4. Modelo, entrenamiento en dos etapas y evaluación

Único cambio frente al Sprint 09: optimizador `AdamW(weight_decay=0.01)` para la cabeza y el fine-tuning. No se cargan pesos de Sprint 09. Se parte otra vez de ImageNet y de una cabeza nueva. Las funciones no entrenan hasta activar la quinta celda.


In [4]:
def construir_modelo(numero_clases, tamano=TAMANO_ENTRADA,
                      volteo_horizontal=False, semilla=SEMILLA, pesos="imagenet"):
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input

    if numero_clases < 2:
        raise ValueError("El clasificador necesita al menos dos clases.")
    tf.keras.utils.set_random_seed(semilla)
    entrada = tf.keras.Input(shape=(*tamano, 3), dtype="float32", name="imagen_rgb")
    x = entrada
    if volteo_horizontal:
        x = tf.keras.layers.RandomFlip("horizontal", seed=semilla, name="aumento_train")(x)
    x = tf.keras.layers.Lambda(preprocess_input, name="preprocesamiento_resnet50")(x)
    modelo_base = tf.keras.applications.ResNet50(
        input_shape=(*tamano, 3), include_top=False, weights=pesos, pooling="avg",
    )
    modelo_base.trainable = False
    x = modelo_base(x, training=False)
    x = tf.keras.layers.Dropout(0.35, seed=semilla)(x)
    salida = tf.keras.layers.Dense(
        numero_clases, activation="softmax", name="clase",
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=semilla),
    )(x)
    modelo = tf.keras.Model(entrada, salida, name="resnet50_palta")
    modelo.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=0.01),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"],
    )
    return modelo


def crear_dataset_evaluacion(tabla):
    """Lectura RGB 0-255 en el orden de tabla, sin aleatoriedad ni aumentos."""
    import tensorflow as tf
    rutas = tabla["archivo"].astype(str).tolist()
    etiquetas = tabla["etiqueta"].to_numpy(dtype=np.int32)
    secuencia = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))

    def leer(ruta_relativa):
        valor = ruta_relativa.item() if hasattr(ruta_relativa, "item") else ruta_relativa
        ruta = valor.decode("utf-8") if isinstance(valor, bytes) else str(valor)
        rgb, _ = ajustar_tamano(cargar_rgb(DATASET / ruta), TAMANO)
        rgb = np.asarray(rgb, dtype=np.float32)
        if rgb.shape != (*TAMANO, 3) or not np.isfinite(rgb).all():
            raise ValueError(f"Imagen inválida en evaluación: {ruta}")
        return rgb

    def mapear(ruta, etiqueta):
        imagen = tf.numpy_function(leer, [ruta], tf.float32)
        imagen.set_shape((*TAMANO, 3))
        etiqueta.set_shape(())
        return imagen, etiqueta

    opciones = tf.data.Options()
    opciones.experimental_deterministic = True
    opciones.threading.private_threadpool_size = 2
    return secuencia.with_options(opciones).map(mapear, num_parallel_calls=1).batch(1).prefetch(1)


def callbacks_etapa(tf, carpeta, etapa):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                         restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            str(carpeta / f"mejor_{etapa}.weights.h5"), monitor="val_loss",
            save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2),
        tf.keras.callbacks.CSVLogger(str(carpeta / f"historial_{etapa}.csv")),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


def evaluar_guardar(modelo, tabla, nombre, carpeta, firma_protocolo):
    """Graba métricas y predicciones una vez, después de fijar el checkpoint."""
    import tensorflow as tf
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        accuracy_score, f1_score, recall_score, precision_score,
        balanced_accuracy_score, classification_report,
        confusion_matrix, ConfusionMatrixDisplay,
        average_precision_score, roc_auc_score, brier_score_loss,
    )
    datos = tabla.reset_index(drop=True)
    secuencia = crear_dataset_evaluacion(datos)
    probabilidades = np.asarray(modelo.predict(secuencia, verbose=0))
    if probabilidades.shape != (len(datos), 2) or not np.isfinite(probabilidades).all():
        raise RuntimeError(f"Predicciones inválidas: {nombre}")
    if not np.allclose(probabilidades.sum(axis=1), 1.0, atol=1e-4):
        raise RuntimeError("Las probabilidades no suman 1.")
    reales = datos["etiqueta"].to_numpy(dtype=np.int32)
    estimadas = np.argmax(probabilidades, axis=1)
    prusset = probabilidades[:, 1]
    informe = classification_report(
        reales, estimadas, labels=[0, 1], target_names=CLASES,
        output_dict=True, zero_division=0,
    )
    metricas = {
        "estado": "completo", "particion": nombre, "n_imagenes": int(len(datos)),
        "firma_protocolo": firma_protocolo,
        "accuracy": float(accuracy_score(reales, estimadas)),
        "macro_f1": float(f1_score(reales, estimadas, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(reales, estimadas)),
        "recall_russet": float(recall_score(reales, estimadas, pos_label=1, zero_division=0)),
        "precision_russet": float(precision_score(reales, estimadas, pos_label=1, zero_division=0)),
        "average_precision_russet": float(average_precision_score(reales, prusset)),
        "roc_auc_russet": float(roc_auc_score(reales, prusset)),
        "brier_russet": float(brier_score_loss(reales, prusset)),
    }
    (carpeta / f"metricas_{nombre}.json").write_text(
        json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    resultados = datos[["archivo", "clase", "etiqueta"]].copy()
    resultados["prediccion"] = [CLASES[int(i)] for i in estimadas]
    resultados["probabilidad_PICADURA"] = probabilidades[:, 0]
    resultados["probabilidad_RUSSET"] = prusset
    resultados.to_csv(carpeta / f"predicciones_{nombre}.csv", index=False)
    pd.DataFrame(informe).transpose().to_csv(carpeta / f"metricas_por_clase_{nombre}.csv")
    matriz = confusion_matrix(reales, estimadas, labels=[0, 1])
    pd.DataFrame(matriz, index=CLASES, columns=CLASES).to_csv(
        carpeta / f"matriz_confusion_{nombre}.csv"
    )
    import matplotlib.pyplot as plt
    vista = ConfusionMatrixDisplay(matriz, display_labels=CLASES).plot(values_format="d")
    vista.ax_.set_title(f"ResNet50 - {nombre}")
    vista.figure_.tight_layout()
    vista.figure_.savefig(carpeta / f"matriz_confusion_{nombre}.png", dpi=150)
    plt.close(vista.figure_)
    print(f"{nombre}: Macro-F1={metricas['macro_f1']:.4f} | "
          f"Recall RUSSET={metricas['recall_russet']:.4f} | "
          f"Accuracy={metricas['accuracy']:.4f}")
    return metricas


def ejecutar_sprint10():
    import tensorflow as tf
    import matplotlib.pyplot as plt
    destino = SALIDA / NOMBRE_CORRIDA
    if destino.exists():
        raise FileExistsError(f"La corrida ya existe y no se sobrescribirá: {destino}")
    if not DATASET.is_dir():
        raise FileNotFoundError(f"No existe DATASET: {DATASET}")

    # Verificar disponibilidad de GPU ANTES de crear carpetas de resultados.
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        raise RuntimeError("No se detectó GPU. Seleccione palta_gpu (CUDA) en WSL2.")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEMILLA)
    try:
        tf.config.experimental.enable_op_determinism()
    except (AttributeError, RuntimeError):
        pass

    # Misma verificación SHA-256 del experimento anterior.
    auditoria_actual = verificar_auditoria(DATASET, aprobar_exclusiones=True)
    for campo in ("sha256_dataset", "sha256_manifiesto"):
        if auditoria_actual[campo] != referencia[campo]:
            raise RuntimeError("Cambió la identidad auditada del dataset.")
    if sorted(auditoria_actual["exclusiones"]) != sorted(referencia["exclusiones"]):
        raise RuntimeError("Las exclusiones difieren del experimento histórico.")

    # Inicio realmente nuevo: backbone ImageNet, cabeza Dense inicializada de cero.
    modelo = construir_modelo(len(CLASES), TAMANO, False, SEMILLA, pesos="imagenet")
    base = next(capa for capa in modelo.layers if isinstance(capa, tf.keras.Model))
    if base.trainable or not all(not c.trainable for c in base.layers):
        raise RuntimeError("La base ResNet50 debe estar congelada en etapa de cabeza.")
    validacion = crear_dataset_evaluacion(CONJUNTOS["validation_original"])
    entrenamiento_cabeza = crear_train_fotometrico(
        train_efectivo, DATASET, TAMANO, lote=1, semilla=SEMILLA, foto_extra=False
    )
    entrenamiento_ft = crear_train_fotometrico(
        train_efectivo, DATASET, TAMANO, lote=1, semilla=SEMILLA, foto_extra=True
    )

    configuracion = {
        "condicion": "RN50_D1_PHOTO_ADAMW_WD001",
        "nombre_ejecucion": NOMBRE_CORRIDA,
        "sprint": 10,
        "referencia_control": "RN50_FECHA260423_s42_01",
        "variable_experimental": "AdamW_versus_Adam",
        "weight_decay_adamw": 0.01,
        "optimizador_cabeza": "AdamW",
        "optimizador_finetune": "AdamW",
        "origen_pesos_backbone": "ImageNet",
        "carga_cabeza_historica": False,
        "entrenamiento_independiente": True,
        "semilla": SEMILLA,
        "clases": CLASES,
        "tamano_entrada": list(TAMANO),
        "train": 529, "validation": 127,
        "evaluacion_exploratoria_260423": 43,
        "train_efectivo_d1": len(train_efectivo),
        "copias_russet": len(plan_d1),
        "pesos_clase": pesos_efectivos,
        "epocas_max_cabeza": 8, "epocas_max_finetune": 8,
        "learning_rate_cabeza": 1e-3, "learning_rate_finetune": 1e-5,
        "dropout": 0.35, "lote": 1,
        "bloque_finetune": "conv5", "batch_normalization": "congelada",
        "fotometria_d1": {"brillo": [0.92, 1.08], "contraste": [0.90, 1.10]},
        "fotometria_extra_finetune": {"brillo": [0.95, 1.05], "contraste": [0.95, 1.05]},
        "seleccion_checkpoint": "min_val_loss en validation_original",
        "sha256_manifiesto_260423": hash_archivo(MANIFIESTO_RUTA),
        "sha256_dataset": auditoria_actual["sha256_dataset"],
        "sha256_manifest_original": auditoria_actual["sha256_manifiesto"],
        "exclusiones_originales": auditoria_actual["exclusiones"],
        "versiones": {nombre: importlib.metadata.version(nombre)
                      for nombre in ("tensorflow", "numpy", "pandas", "Pillow", "scikit-learn")},
        "python": platform.python_version(),
        "limitacion": "260423 es exploratorio y no demuestra independencia por fruto/lote/campaña.",
    }
    firma = hashlib.sha256(json.dumps(configuracion, sort_keys=True).encode()).hexdigest()
    configuracion["firma_protocolo"] = firma

    destino.mkdir(parents=True, exist_ok=False)
    (destino / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")
    (destino / "auditoria.json").write_text(
        json.dumps(auditoria_actual, ensure_ascii=False, indent=2), encoding="utf-8")
    manifiesto.to_csv(destino / "manifiesto_260423_usado.csv", index=False)
    plan_d1.to_csv(destino / "copias_russet_train.csv", index=False)
    train_efectivo.to_csv(destino / "manifiesto_train_efectivo.csv", index=False)
    estado = destino / "estado.json"
    estado.write_text(json.dumps({"estado": "iniciado"}), encoding="utf-8")
    inicio = time.perf_counter()

    try:
        # Etapa 1: cabeza totalmente nueva, con ResNet50 ImageNet congelada.
        historial_cabeza = modelo.fit(
            entrenamiento_cabeza, steps_per_epoch=len(train_efectivo),
            validation_data=validacion, epochs=8, class_weight=pesos_efectivos,
            callbacks=callbacks_etapa(tf, destino, "cabeza"), verbose=1,
        )
        checkpoint_cabeza = destino / "mejor_cabeza.weights.h5"
        if not checkpoint_cabeza.is_file():
            raise RuntimeError("Falta el checkpoint de la cabeza nueva.")
        modelo.load_weights(str(checkpoint_cabeza))

        # Etapa 2: el mismo fine-tuning de conv5, BatchNormalization congelada.
        base.trainable = True
        for capa in base.layers:
            capa.trainable = capa.name.startswith("conv5_") and not isinstance(
                capa, tf.keras.layers.BatchNormalization
            )
        if sum(int(capa.trainable) for capa in base.layers) < 1:
            raise RuntimeError("No se encontraron capas conv5 ajustables.")
        modelo.compile(
            optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=0.01),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"],
        )
        historial_ft = modelo.fit(
            entrenamiento_ft, steps_per_epoch=len(train_efectivo),
            validation_data=validacion, epochs=8, class_weight=pesos_efectivos,
            callbacks=callbacks_etapa(tf, destino, "finetune"), verbose=1,
        )
        checkpoint_ft = destino / "mejor_finetune.weights.h5"
        if not checkpoint_ft.is_file():
            raise RuntimeError("Falta el checkpoint del fine-tuning.")
        modelo.load_weights(str(checkpoint_ft))

        # Selección por validation_original. Este sprint NO evalúa 260423.
        metricas_val = evaluar_guardar(
            modelo, CONJUNTOS["validation_original"], "validation_original", destino, firma
        )
        # 260423 ya se consultó en el Sprint 09 y NO se reutiliza para
        # optimizar este experimento. TEST tampoco se evalúa.

        for etapa, historial in (("cabeza", historial_cabeza), ("finetune", historial_ft)):
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(historial.history["loss"], label="train")
            ax.plot(historial.history["val_loss"], label="validation")
            ax.set(xlabel="Época", ylabel="Loss", title=f"Curvas {etapa}")
            ax.legend()
            fig.tight_layout()
            fig.savefig(destino / f"curva_loss_{etapa}.png", dpi=150)
            plt.close(fig)

        resumen = (
            "# Resultados Sprint 10: regularización controlada con AdamW\n\n"
            "Se entrenó una nueva cabeza desde ImageNet con exactamente los mismos "
            "conjuntos, D1, semilla y tasas del Sprint 09. El único cambio es AdamW "
            "(weight_decay=0.01) en ambas etapas.\n\n"
            f"- VALIDATION original (127): Macro-F1={metricas_val['macro_f1']:.4f}, "
            f"recall RUSSET={metricas_val['recall_russet']:.4f}, "
            f"precisión RUSSET={metricas_val['precision_russet']:.4f}.\n\n"
            "**Limitaciones:** la comparación usa el conjunto de desarrollo ya consultado "
            "en los sprints anteriores. La diferencia de una sola semilla no es evidencia "
            "estadística de mejora. No se ha consultado 260423 ni TEST en esta corrida.\n"
        )
        (destino / "resumen_resultados.md").write_text(resumen, encoding="utf-8")
        estado.write_text(json.dumps({
            "estado": "completo",
            "epocas_cabeza": len(historial_cabeza.history["loss"]),
            "mejor_epoca_cabeza": int(np.argmin(historial_cabeza.history["val_loss"])) + 1,
            "epocas_finetune": len(historial_ft.history["loss"]),
            "mejor_epoca_finetune": int(np.argmin(historial_ft.history["val_loss"])) + 1,
            "segundos_transcurridos": round(time.perf_counter() - inicio, 2),
            "evaluacion_exploratoria_consultada": False,
            "test_consultado": False,
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Corrida completa:", destino.relative_to(RAIZ))
        print("No se han calculado predicciones ni métricas de 260423 o TEST.")
    except (Exception, KeyboardInterrupt) as error:
        import traceback
        estado.write_text(json.dumps({
            "estado": "incompleto", "detalle": str(error),
            "traceback": traceback.format_exc(),
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        raise


## 5. Ejecutar una única corrida AdamW

**El control comienza en `False`.** La ejecución necesita el kernel `palta_gpu (CUDA)` y tarda más que las comprobaciones. No usar `Run All`.


In [ ]:
# CELDA DE CONTROL. Por defecto no entrena.
EJECUTAR_EXPERIMENTO = False

if EJECUTAR_EXPERIMENTO:
    ejecutar_sprint10()
    print("Desactive EJECUTAR_EXPERIMENTO antes de guardar el notebook.")
else:
    print("Modo seguro. La corrida", NOMBRE_CORRIDA, "no se ha ejecutado desde esta celda.")
    print("Para una ejecución controlada, cambiar a True una sola vez.")


I0000 00:00:1794179494.795364  772868 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794179494.955023  772868 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794179496.703390  772868 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Integridad: 100/829
Integridad: 200/829
Integridad: 300/829
Integridad: 400/829
Integridad: 500/829
Integridad: 600/829
Integridad: 700/829
Integridad: 800/829
Integridad: 829/829


I0000 00:00:1794179530.835244  772868 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5561 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/8


/home/tci_auto/.venvs/palta_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1794179540.162131  780002 cuda_dnn.cc:461] Loaded cuDNN version 92700


584/584 ━━━━━━━━━━━━━━━━━━━━ 101s 163ms/step - accuracy: 0.6849 - loss: 0.7670 - val_accuracy: 0.2677 - val_loss: 1.7774 - learning_rate: 0.0010
Epoch 2/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 93s 159ms/step - accuracy: 0.7877 - loss: 0.4593 - val_accuracy: 0.8425 - val_loss: 0.3877 - learning_rate: 0.0010
Epoch 3/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 92s 158ms/step - accuracy: 0.8134 - loss: 0.4699 - val_accuracy: 0.8819 - val_loss: 0.2594 - learning_rate: 0.0010
Epoch 4/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 95s 162ms/step - accuracy: 0.8476 - loss: 0.3519 - val_accuracy: 0.8976 - val_loss: 0.2077 - learning_rate: 0.0010
Epoch 5/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 92s 157ms/step - accuracy: 0.8596 - loss: 0.3208 - val_accuracy: 0.9134 - val_loss: 0.2260 - learning_rate: 0.0010
Epoch 6/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 89s 152ms/step - accuracy: 0.8493 - loss: 0.3682 - val_accuracy: 0.8976 - val_loss: 0.2481 - learning_rate: 0.0010
Epoch 7/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 91s 156ms/step - accuracy: 0.9144 - loss: 0.21

## 6. Comparación de resultados de VALIDATION (sin entrenar)

Lee las métricas guardadas de Sprint 09 y Sprint 10, únicamente del conjunto `validation_original` de 127 imágenes. No consulta `260423` ni TEST y no escoge un modelo automáticamente.


In [7]:
# Consulta de métricas existentes. No entrena ni evalúa imágenes.
carpeta = SALIDA / NOMBRE_CORRIDA
archivo_estado = carpeta / "estado.json"
if not archivo_estado.is_file():
    print("La corrida Sprint 10 aún no existe. No se ha entrenado nada desde esta celda.")
else:
    estado_actual = json.loads(archivo_estado.read_text(encoding="utf-8"))
    print("Estado:", estado_actual.get("estado"))
    if estado_actual.get("estado") == "completo":
        referencia = RAIZ / "7. resultados/validacion_por_fecha/RN50_FECHA260423_s42_01/metricas_validation_original.json"
        candidato = carpeta / "metricas_validation_original.json"
        if not referencia.is_file():
            raise FileNotFoundError(f"Falta la referencia del Sprint 09: {referencia}")
        filas = []
        for nombre, ruta in (("Sprint 09 - Adam", referencia), ("Sprint 10 - AdamW", candidato)):
            met = json.loads(ruta.read_text(encoding="utf-8"))
            filas.append({
                "modelo": nombre, "n": met["n_imagenes"],
                "accuracy": met["accuracy"], "macro_f1": met["macro_f1"],
                "precision_russet": met["precision_russet"],
                "recall_russet": met["recall_russet"],
                "average_precision_russet": met["average_precision_russet"],
                "brier_russet": met["brier_russet"],
            })
        comparacion = pd.DataFrame(filas)
        display(comparacion.round(4))
        columnas = ["accuracy", "macro_f1", "precision_russet", "recall_russet",
                    "average_precision_russet", "brier_russet"]
        cambios = (comparacion.iloc[1][columnas] - comparacion.iloc[0][columnas]).to_frame("diferencia_AdamW_menos_Adam")
        display(cambios.round(4))
        print("Comparación descriptiva de una sola semilla sobre VALIDATION; no prueba generalización.")
        print("No se calcularon métricas ni predicciones de 260423 o TEST.")
    else:
        print("Ejecución incompleta. Consulte estado.json sin sobrescribir el experimento.")


Estado: completo


,modelo,n,accuracy,macro_f1,precision_russet,recall_russet,average_precision_russet,brier_russet
0,Sprint 09 - Adam,127,0.8819,0.7900,0.6364,0.6667,0.8268,0.0669
1,Sprint 10 - AdamW,127,0.8898,0.7923,0.6842,0.6190,0.8333,0.0648


,diferencia_AdamW_menos_Adam
accuracy,0.007874
macro_f1,0.002253
precision_russet,0.047847
recall_russet,-0.047619
average_precision_russet,0.006435
brier_russet,-0.002115


Comparación descriptiva de una sola semilla sobre VALIDATION; no prueba generalización.
No se calcularon métricas ni predicciones de 260423 o TEST.


## Interpretación y límites de la mejora

- El único factor cambiado respecto al Sprint 09 es el algoritmo de optimización (Adam → AdamW) con regularización desacoplada (`weight_decay=0.01`) en la cabeza y `conv5`. El resto de parámetros, datos y semilla deben ser idénticos.
- **Criterio exploratorio de mejora:** examinar si sube Macro-F1 y/o precisión RUSSET en VALIDATION sin deteriorar considerablemente el recall. La decisión no se basa en `260423` ni en TEST.
- El grupo `260423` ya se consultó en el Sprint 09. No se usa para entrenar, seleccionar parámetros ni estimar de nuevo una mejora en este sprint.
- La validación original se ha utilizado reiteradamente; una mejora en 127 imágenes puede ser optimismo por selección. Para una afirmación más sólida se requieren repeticiones con diferentes semillas y evaluación externa nueva o validación por grupos cuidadosamente diseñada.
- Los checkpoints `*.weights.h5` quedan locales; no publicar ni sobrescribir la corrida del Sprint 09.
